# Stage 3 – DistilBERT fine-tuning
**Owner: Kunal Parmar** · ICT6001 Applied Project

Fine-tunes `distilbert-base-uncased` (Hugging Face Transformers + PyTorch). The learning rate is chosen on a stratified 30% subsample, then the final model trains on the full training set for up to 3 epochs with early stopping on validation F1. A checkpoint is saved to Drive after every epoch.

**Runtime → Change runtime type → T4 GPU.** About 30–50 minutes. If Colab disconnects, run all cells again: training resumes from the last checkpoint.

In [ ]:
# Setup: connect Google Drive and point to the project folder
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/phishing_project'

import os, sys, json, time, random, pickle, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sys.path.insert(0, PROJECT_DIR)
warnings.filterwarnings('ignore')

SEED = 42
DATA_DIR  = os.path.join(PROJECT_DIR, 'data')
OUT_DIR   = os.path.join(PROJECT_DIR, 'outputs')
SPLIT_DIR = os.path.join(OUT_DIR, 'splits')
MODEL_DIR = os.path.join(OUT_DIR, 'models')
FIG_DIR   = os.path.join(OUT_DIR, 'figures')
RUN_LOG   = os.path.join(OUT_DIR, 'run_log.csv')
for d in (OUT_DIR, SPLIT_DIR, MODEL_DIR, FIG_DIR): os.makedirs(d, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
sns.set_theme(style='whitegrid')
C_SVM, C_BERT, C_PHISH, C_LEGIT = '#3B6EA5', '#D9822B', '#C8553D', '#7FB77E'

def savefig(name):
    plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, name), dpi=200, bbox_inches='tight'); plt.show()
    print('Saved figure:', name)
def save_summary(stage, d):
    with open(os.path.join(OUT_DIR, f'summary_{stage}.json'), 'w') as f: json.dump(d, f, indent=1, default=str)
print('Project folder:', PROJECT_DIR)

In [ ]:
!pip -q install accelerate

In [ ]:
import torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, EarlyStoppingCallback, DataCollatorWithPadding)
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from src.run_log import log_run
torch.manual_seed(SEED)
MODEL_NAME, MAX_LEN, BATCH = 'distilbert-base-uncased', 256, 16
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU - change the runtime to T4 GPU before continuing')
TMP_DIR = '/content/tmp' if os.path.isdir('/content') else os.path.join(OUT_DIR, 'tmp')

## 3.1 Load the splits and check email length in tokens

In [ ]:
train = pd.read_csv(os.path.join(SPLIT_DIR, 'train.csv')); val = pd.read_csv(os.path.join(SPLIT_DIR, 'val.csv'))
for d in (train, val): d['clean'] = d['clean'].fillna('')
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
sample = train['clean'].sample(min(5000, len(train)), random_state=SEED)
lens = np.array([len(x) for x in tok(list(sample), truncation=False)['input_ids']])
token_stats = {'median': int(np.median(lens)), 'p90': int(np.percentile(lens, 90)), 'share_over_256': round(float((lens > MAX_LEN).mean()), 4)}
print(token_stats)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(np.clip(lens, 0, 1024), bins=50, color=C_BERT, alpha=.85); ax.axvline(MAX_LEN, color='black', ls='--', label='256-token limit')
ax.set_title('Email length in DistilBERT tokens (capped at 1,024)'); ax.set_xlabel('Tokens'); ax.set_ylabel('Emails'); ax.legend()
savefig('fig06_token_lengths.png')

## 3.2 Tokenise and set up training

In [ ]:
class EmailDS(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.enc = tok(list(texts), truncation=True, max_length=MAX_LEN); self.labels = list(labels)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}; item['labels'] = torch.tensor(self.labels[i]); return item

start = time.time()
train_ds, val_ds = EmailDS(train['clean'], train['label']), EmailDS(val['clean'], val['label'])
print(f'Tokenised {len(train_ds):,} training and {len(val_ds):,} validation emails in {time.time() - start:.0f}s')
collator = DataCollatorWithPadding(tok)

def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    return {'f1': f1_score(p.label_ids, preds), 'accuracy': accuracy_score(p.label_ids, preds),
            'precision': precision_score(p.label_ids, preds), 'recall': recall_score(p.label_ids, preds)}
def new_model():
    return AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2,
        id2label={0: 'legitimate', 1: 'phishing'}, label2id={'legitimate': 0, 'phishing': 1})
def make_args(out, lr, epochs, n_train):
    steps = int(np.ceil(n_train / BATCH)) * epochs
    return TrainingArguments(output_dir=out, learning_rate=lr, num_train_epochs=epochs, per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=64, weight_decay=0.01, warmup_steps=int(0.06 * steps), eval_strategy='epoch',
        save_strategy='epoch', load_best_model_at_end=True, metric_for_best_model='f1', greater_is_better=True,
        save_total_limit=2, fp16=(DEVICE == 'cuda'), seed=SEED, logging_steps=100, report_to='none')

## 3.3 Learning-rate search (1 epoch each on a 30% subsample)

In [ ]:
sub = train.groupby('label').sample(frac=0.30, random_state=SEED)
sub_ds = EmailDS(sub['clean'], sub['label'])
lr_results = []
for lr in [2e-5, 3e-5]:
    start = time.time()
    t = Trainer(model=new_model(), args=make_args(os.path.join(TMP_DIR, f'lr_{lr}'), lr, 1, len(sub_ds)),
                train_dataset=sub_ds, eval_dataset=val_ds, data_collator=collator, compute_metrics=compute_metrics)
    t.train(); ev = t.evaluate()
    lr_results.append({'learning_rate': lr, 'val_f1': round(ev['eval_f1'], 4), 'minutes': round((time.time() - start) / 60, 1)})
    del t
    if DEVICE == 'cuda': torch.cuda.empty_cache()
lr_results = pd.DataFrame(lr_results); print(lr_results)
BEST_LR = float(lr_results.loc[lr_results['val_f1'].idxmax(), 'learning_rate']); print('Best learning rate =', BEST_LR)

## 3.4 Final fine-tuning on the full training set

In [ ]:
CKPT_DIR = os.path.join(MODEL_DIR, 'distilbert_checkpoints')
trainer = Trainer(model=new_model(), args=make_args(CKPT_DIR, BEST_LR, 3, len(train_ds)), train_dataset=train_ds,
                  eval_dataset=val_ds, data_collator=collator, compute_metrics=compute_metrics,
                  callbacks=[EarlyStoppingCallback(early_stopping_patience=1)])
resume = any(n.startswith('checkpoint-') for n in os.listdir(CKPT_DIR)) if os.path.isdir(CKPT_DIR) else False
print('Resuming from the last checkpoint' if resume else 'Starting training')
start = time.time()
trainer.train(resume_from_checkpoint=resume)
train_seconds = time.time() - start
FINAL_DIR = os.path.join(MODEL_DIR, 'distilbert_final')
trainer.save_model(FINAL_DIR); tok.save_pretrained(FINAL_DIR)
size_mb = sum(os.path.getsize(os.path.join(FINAL_DIR, f)) for f in os.listdir(FINAL_DIR)) / 1e6
print(f'Training finished in {train_seconds / 60:.1f} min | model saved ({size_mb:.0f} MB)')

In [ ]:
hist = pd.DataFrame(trainer.state.log_history)
hist.to_csv(os.path.join(OUT_DIR, 'distilbert_training_log.csv'), index=False)
ev = hist.dropna(subset=['eval_f1'])[['epoch', 'eval_loss', 'eval_f1', 'eval_precision', 'eval_recall']]
print(ev.round(4).to_string(index=False))
tl = hist.dropna(subset=['loss']) if 'loss' in hist else pd.DataFrame()
fig, ax = plt.subplots(figsize=(7, 3.5))
if len(tl): ax.plot(tl['epoch'], tl['loss'], color=C_BERT, alpha=.6, label='Training loss')
ax.plot(ev['epoch'], ev['eval_loss'], 'o-', color='black', label='Validation loss')
ax.set_title('DistilBERT training and validation loss'); ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.legend()
savefig('fig07_distilbert_loss.png')

## 3.5 Log the run

In [ ]:
summary = {'model': MODEL_NAME, 'max_len': MAX_LEN, 'batch_size': BATCH, 'token_lengths': token_stats,
           'lr_search': lr_results.to_dict(orient='records'), 'best_lr': BEST_LR, 'resumed': resume,
           'train_minutes_this_session': round(train_seconds / 60, 1), 'epochs_run': float(ev['epoch'].max()),
           'val_by_epoch': ev.round(4).to_dict(orient='records'), 'best_val_f1': round(float(ev['eval_f1'].max()), 4),
           'size_mb': round(size_mb, 1), 'device': torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'}
save_summary('03_distilbert', summary)
log_run(RUN_LOG, 'train', 'DistilBERT', {'lr': BEST_LR, 'batch': BATCH, 'max_len': MAX_LEN, 'max_epochs': 3, 'early_stopping': 1},
        {'best_val_f1': summary['best_val_f1'], 'epochs_run': summary['epochs_run']}, 'Kunal')
print('Stage 3 complete.')